# Sprint 3: Query Processing & Document Retrieval

This notebook demonstrates the Sprint 3 retrieval pipeline end to end.
It validates:
- Query embedding generation
- Similarity search against the vector database
- Retrieval of relevant documents with metadata
- Retrieval endpoint behavior before generation is added

## Setup

Initialize imports, resolve the repository root, and prepare shared helpers for the API calls used throughout the notebook.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path
from urllib import parse, request
from urllib.error import URLError

def find_repo_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "pyproject.toml").exists():
        return cwd
    if (cwd.parent / "pyproject.toml").exists():
        return cwd.parent
    raise RuntimeError("Could not locate repo root with pyproject.toml")

def post_json(url: str, payload: dict, timeout: int = 30) -> dict:
    req = request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))

ROOT = find_repo_root()
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PORT = 8016
BASE_URL = f"http://127.0.0.1:{PORT}"
server_proc = None
auth_env = {}

print(f"Repo root: {ROOT}")
print(f"Base URL: {BASE_URL}")

## 1) Authenticate with AI Lab

Authenticate directly from Python and keep the bearer token in notebook state so the API server can use it without requiring a separate CLI login step.

In [ ]:
from datetime import datetime
from ailab.utils.azure import authenticate_ailab_interactive

auth_result = authenticate_ailab_interactive()
auth_env = {"AILAB_BEARER_TOKEN": auth_result["token"]}

expires_on = datetime.fromtimestamp(auth_result["expires_on"])
print(f"Authenticated via {auth_result['auth_method']}")
print(f"Token expires at: {expires_on}")

## 2) Inspect Notebook Auth State

Confirm the notebook has a bearer token before starting the FastAPI server.

In [ ]:
auth_status = {
    "has_token": bool(auth_env.get("AILAB_BEARER_TOKEN")),
    "token_prefix": auth_env.get("AILAB_BEARER_TOKEN", "")[:16] + "..." if auth_env.get("AILAB_BEARER_TOKEN") else None,
}

print(json.dumps(auth_status, indent=2))

if not auth_status["has_token"]:
    raise RuntimeError("Authentication token is missing. Re-run the authentication cell.")

## 3) Start FastAPI Server

Launch the FastAPI server and pass the notebook-acquired bearer token into the server environment.

In [ ]:
if server_proc is not None and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server_proc.kill()

cmd = [
    sys.executable,
    "-m",
    "uvicorn",
    "main:app",
    "--app-dir",
    str(SRC_DIR),
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
]

server_env = os.environ.copy()
server_env.update(auth_env)

server_proc = subprocess.Popen(
    cmd,
    cwd=ROOT,
    env=server_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

ready = False
for _ in range(60):
    try:
        with request.urlopen(f"{BASE_URL}/health", timeout=1) as resp:
            if resp.status == 200:
                ready = True
                break
    except URLError:
        time.sleep(0.5)

if not ready or server_proc.poll() is not None:
    stderr_output = ""
    if server_proc.stderr is not None:
        stderr_output = server_proc.stderr.read()
    raise RuntimeError(f"Server failed to start. STDERR: {stderr_output}")

print("Server started successfully")

## 4) Verify Server Auth Status

Confirm the FastAPI process received authentication material before retrieval begins.

In [ ]:
with request.urlopen(f"{BASE_URL}/auth/status", timeout=5) as resp:
    server_auth_status = json.loads(resp.read().decode("utf-8"))

print(json.dumps(server_auth_status, indent=2))

if not server_auth_status["authenticated"]:
    raise RuntimeError("Server does not have usable authentication material. Re-run the authentication cell before continuing.")

## 5) Check Vector Database Status

Inspect the vector database before ingestion to show the retrieval pipeline starting state.

In [ ]:
with request.urlopen(f"{BASE_URL}/vectordb/status", timeout=5) as resp:
    db_status = json.loads(resp.read().decode("utf-8"))

print("Vector DB Status (before ingestion):")
print(json.dumps(db_status, indent=2))

## 6) Ingest Documents

Load a small sample of source documents so the retrieval steps have content to search.

In [ ]:
ingest_result = post_json(
    f"{BASE_URL}/vectordb/ingest",
    {"limit": 5},
    timeout=300,
)

print("Ingestion Result:")
print(json.dumps(ingest_result, indent=2))

## 7) Generate Query Embedding

Call the explicit query embedding endpoint to make the vectorization step observable.

In [ ]:
query_payload = {
    "query": "What is artificial intelligence?",
    "top_k": 3,
}

embedding_result = post_json(
    f"{BASE_URL}/vectordb/query/embedding",
    query_payload,
    timeout=30,
)

print("Query Embedding Response:")
print(json.dumps({
    "query": embedding_result["query"],
    "dimensions": embedding_result["dimensions"],
    "embedding_preview": embedding_result["embedding"][:8],
}, indent=2))

## 8) Retrieve Relevant Documents

Call the structured retrieval endpoint to embed the query, run similarity search, and return relevant documents plus metadata.

In [ ]:
retrieval_result = post_json(
    f"{BASE_URL}/vectordb/query",
    query_payload,
    timeout=30,
)

print("Structured Retrieval Response:")
print(json.dumps({
    "query": retrieval_result["query"],
    "top_k": retrieval_result["top_k"],
    "embedding_dimensions": len(retrieval_result["embedding"]),
    "result_count": len(retrieval_result["results"]),
}, indent=2))

if retrieval_result["results"]:
    print("\nFirst Retrieved Document:")
    print(json.dumps(retrieval_result["results"][0], indent=2))
else:
    print("No retrieval results were returned.")

## 9) Compare Legacy GET Query Endpoint

Call the original GET query endpoint to show compatibility with the new explicit retrieval flow.

In [ ]:
legacy_query = parse.urlencode({
    "q": query_payload["query"],
    "top_k": query_payload["top_k"],
})

with request.urlopen(f"{BASE_URL}/vectordb/query?{legacy_query}", timeout=30) as resp:
    legacy_results = json.loads(resp.read().decode("utf-8"))

print(f"Legacy GET result count: {len(legacy_results)}")
if legacy_results:
    print(json.dumps(legacy_results[0], indent=2))

## 10) Verify Retrieval Endpoints

Check the key retrieval-related endpoints and confirm that they respond successfully.

In [ ]:
checks = [
    ("/health", "GET", None, 5),
    ("/auth/status", "GET", None, 5),
    ("/vectordb/status", "GET", None, 5),
    ("/vectordb/query/embedding", "POST", query_payload, 30),
    ("/vectordb/query", "POST", query_payload, 30),
]

for endpoint, method, payload, timeout in checks:
    try:
        if method == "GET":
            with request.urlopen(f"{BASE_URL}{endpoint}", timeout=timeout) as resp:
                status_code = resp.status
        else:
            _ = post_json(f"{BASE_URL}{endpoint}", payload, timeout=timeout)
            status_code = 200
        print(f"OK  {method:4s} {endpoint:26s} -> {status_code}")
    except Exception as exc:
        print(f"ERR {method:4s} {endpoint:26s} -> {exc}")

## 11) Cleanup

Stop the test server started by this notebook.

In [ ]:
if server_proc is not None and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server_proc.kill()

print("Server process cleaned up")

## Summary

Sprint 3 now demonstrates:
- Query embedding generation via `/vectordb/query/embedding`
- Structured retrieval via `POST /vectordb/query`
- Backward-compatible retrieval via `GET /vectordb/query`
- Retrieval of relevant documents and metadata from the vector database

<!-- re-push -->